In [1]:
import psycopg2
import pandas as pd
import numpy as np
import requests 
from bs4 import BeautifulSoup 
from io import StringIO
import matplotlib.pyplot as plt
import os 
import pickle 
import json
from datetime import datetime
import time
import psycopg2
from flask import jsonify

conn = psycopg2.connect(
    host="localhost",
    database="ncaa",
    port = 5432
)
cursor = conn.cursor()

cursor.execute("SELECT * FROM players_box_w_conferences;")
cols = [desc[0] for desc in cursor.description]
players_box_df = pd.DataFrame(cursor.fetchall(), columns=cols)

cursor.execute("SELECT * FROM matchups_w_conferences;")
cols = [desc[0] for desc in cursor.description]
matchups_df = pd.DataFrame(cursor.fetchall(), columns=cols)

cursor.execute("SELECT DISTINCT conference FROM conferences;")
conferences = {row[0] for row in cursor.fetchall()}

cursor.execute("SELECT * FROM current_ap;")
ap_poll = pd.DataFrame(cursor.fetchall(), columns=["Team", "rank"])

team_name_mapping = {
        "Connecticut": "UConn",
        "North Carolina": "UNC",
        "St. John's": "St. John's (NY)"
    }

ap_poll["Team"] = ap_poll["Team"].map(
    lambda x : team_name_mapping.get(x, x)
)

def get_conference_standings(conference):
    # ------------------------------------------------------------------
    # Validate conference
    # ------------------------------------------------------------------
    if conference not in conferences:
        raise ValueError("conference is not in the list of accepted conferences")

    # ------------------------------------------------------------------
    # Fetch matchups involving the conference
    # ------------------------------------------------------------------
    conf_games = (
        matchups_df[
            (matchups_df["team_conference"] == conference) |
            (matchups_df["opp_conference"] == conference)
        ]
        .copy()
    )

    # ------------------------------------------------------------------
    # Identify conference vs conference games
    # ------------------------------------------------------------------
    conf_games["conference_game"] = (
        (conf_games["team_conference"] == conference) &
        (conf_games["opp_conference"] == conference)
    )

    # ------------------------------------------------------------------
    # Expand to two rows per game (team + opponent perspectives)
    # ------------------------------------------------------------------
    team_rows = pd.DataFrame({
        "Team": conf_games["team"],
        "Conference": conf_games["team_conference"],
        "win": conf_games["team_win"],
        "conference_game": conf_games["conference_game"],
    })

    opp_rows = pd.DataFrame({
        "Team": conf_games["opponent"],
        "Conference": conf_games["opp_conference"],
        "win": conf_games["opponent_win"],
        "conference_game": conf_games["conference_game"],
    })

    all_rows = pd.concat([team_rows, opp_rows], ignore_index=True)

    # Keep only teams in this conference
    conf_only = all_rows[all_rows["Conference"] == conference]

    # ------------------------------------------------------------------
    # Aggregate standings
    # ------------------------------------------------------------------
    standings = conf_only.groupby("Team").agg(
        wins=("win", "sum"),
        losses=("win", lambda x: (1 - x).sum()),
        conference_wins=(
            "win",
            lambda x: x[conf_only.loc[x.index, "conference_game"]].sum()
        ),
        conference_losses=(
            "win",
            lambda x: (1 - x)[conf_only.loc[x.index, "conference_game"]].sum()
        )
    ).reset_index()

    # ------------------------------------------------------------------
    # Merge AP Poll (explicit columns for safety)
    # ------------------------------------------------------------------
    standings = standings.merge(ap_poll, on="Team", how="left")
    standings["rank"] = standings["rank"].fillna("NR")

    # ------------------------------------------------------------------
    # Compute win percentages (NUMERIC)
    # ------------------------------------------------------------------
    standings["conf_win_pct"] = (
        standings["conference_wins"] /
        (standings["conference_wins"] + standings["conference_losses"])
    ).fillna(0)

    standings["overall_win_pct"] = (
        standings["wins"] /
        (standings["wins"] + standings["losses"])
    ).fillna(0)

    # ------------------------------------------------------------------
    # Sort FIRST (important: numeric sorting)
    # ------------------------------------------------------------------
    standings = standings.sort_values(
        ["conf_win_pct", "conference_losses", "overall_win_pct", "wins", "rank"],
        ascending=[False, True, False, False, True]
    ).reset_index(drop=True)

    # ------------------------------------------------------------------
    # Format percentages LAST
    # ------------------------------------------------------------------
    standings["conf_win_pct"] = standings["conf_win_pct"].map("{:.3f}".format)
    standings["overall_win_pct"] = standings["overall_win_pct"].map("{:.3f}".format)

    # ------------------------------------------------------------------
    # Final output
    # ------------------------------------------------------------------
    return standings[
        [
            "rank",
            "Team",
            "wins",
            "losses",
            "overall_win_pct",
            "conference_wins",
            "conference_losses",
            "conf_win_pct",
        ]
    ]

def get_top_25():
    # ------------------------------------------------------------
    # Teams / Conferences
    # ------------------------------------------------------------
    cursor.execute("SELECT team, conference FROM teams_w_conferences;")
    teams_conferences = pd.DataFrame(cursor.fetchall(), columns=["Team", "Conference"])

    result = ap_poll.merge(
        teams_conferences.drop_duplicates(),
        on="Team",
        how="left"
    )

    # ------------------------------------------------------------
    # Records (team_win / team_loss already exist)
    # ------------------------------------------------------------
    matchup_results = matchups_df[["team", "team_win", "team_loss"]]

    records = matchup_results.groupby("team").agg(
        W=("team_win", "sum"),
        L=("team_loss", "sum")
    ).reset_index().rename(columns={"team" : "Team"})

    result = result.merge(records, on="Team", how="left")

    # ------------------------------------------------------------
    # Final output
    # ------------------------------------------------------------
    final_result = (
        result.rename(columns={"rank" : "Current Rank"})
        [["Current Rank", "Team", "Conference", "W", "L"]]
        .drop_duplicates()
        .sort_values("Current Rank")
        .reset_index(drop=True)
    )

    return final_result.where(pd.notna(final_result), None)

def get_player_pg_stats(player, team=None):

    if player not in players_box_df["player"].unique():
        raise Exception("player not found in data")
    
    player_games = players_box_df[players_box_df.player == player]
    
    # If team is specified, filter to that team
    if team:
        player_games = player_games[player_games.team == team]
    
    if len(player_games) == 0:
        raise Exception(f"player {player} not found on team {team}")
    
    player_team = player_games.iloc[0]["team"]
    conference = player_games.iloc[0]["conference"]              
    pg_stats = (
        player_games
        .drop(["gameid", "date", "team", "opponent", "result", "role", "conference", "venue"], axis=1)
        .groupby("player")
        .agg("mean")
    )

    pg_stats["player"] = player
    pg_stats["team"] = player_team
    pg_stats["conference"] = conference
    pg_stats["gp"] = len(player_games[player_games.mp > 0])
    pg_stats["gs"] = len(player_games[player_games.role == "Starter"])

    col_order = ["player", "team", "conference", "gp", "gs", "mp", "pts", "fgm", "fga", "fg_pct", 
                 "fg2m", "fg2a", "fg2_pct", "fg3m", "fg3a", "fg3_pct", "ftm", "fta", "ft_pct", "orb", "drb", 
                 "trb", "ast", "stl", "blk", "tov", "pf", "gmsc"]
    
    return pg_stats[col_order].reset_index(drop=True)

def get_players_pg_stats(players):
    df = pd.DataFrame()
    for player in players:
        df = pd.concat([df, get_player_pg_stats(player)], axis=0)
    return df 

def compare_players(df, players, stats=None, figsize=(14,6), title=None):
    for p in players:
        if p not in df["player"].values:
            raise ValueError(f"{p} not found in DataFrame")

    sub = df[df["player"].isin(players)].set_index("player")

    if stats is None:
        stats = sub.select_dtypes(include=[np.number]).columns.tolist()

    pct_stats = [col for col in stats if col.endswith("_pct") or col.endswith("%")]
    raw_stats = [col for col in stats if col not in pct_stats]

    def _plot_category(stat_list, title_suffix):
        if len(stat_list) == 0:
            return

        values = sub[stat_list].astype(float)
        n_players = len(players)

        x = np.arange(len(stat_list))
        width = 0.8 / n_players

        fig, ax = plt.subplots(figsize=figsize)

        bar_containers = []

        colors = ["lightblue", "pink", "violet", "pastelgreen"]
        for i, player in enumerate(players):
            bars = ax.bar(
                x + (i - n_players/2) * width + width/2,
                values.loc[player],
                width,
                label=player,
                ec="black",
                color=colors[i]
            )
            bar_containers.append((player, bars))

        max_val = values.max().max()
        ax.set_ylim(0, 1.5 * max_val)

        for player, bars in bar_containers:
            for rect in bars:
                height = rect.get_height()
                ax.annotate(
                    f"{height:.2f}",  
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 4),  
                    textcoords="offset points",
                    ha="center",
                    va="bottom",
                    fontsize=9
                )

        ax.set_xticks(x)
        ax.set_xticklabels(stat_list, rotation=45, ha="right")
        ax.set_ylabel("Stat Value")

        final_title = (title or "Player Comparison") + f" — {title_suffix}"
        ax.set_title(final_title)
        ax.legend()

        plt.tight_layout()
        plt.show()

    _plot_category(raw_stats, "Raw Stats")

    _plot_category(pct_stats, "Percentage Stats")

def get_conference_player_pg_stat_leaders(conference, stat, n=25):
    if conference not in conferences:
        raise Exception("conference is not valid")

    if stat not in players_box_df.columns:
        raise Exception("stat is not valid")

    df = players_box_df[players_box_df["conference"] == conference].copy()

    # ------------------------------------------------
    # 1. Games played per player
    # ------------------------------------------------
    games_df = (
        df.groupby(["player", "team"])
        .agg(games_played=("gameid", "nunique"))
        .reset_index()
    )

    # ------------------------------------------------
    # 2. Team games played
    # ------------------------------------------------
    team_games_df = (
        df.groupby("team")
        .agg(team_games=("gameid", "nunique"))
        .reset_index()
    )

    games_df = games_df.merge(team_games_df, on="team", how="left")

    games_df["qualifies_games"] = (
        games_df["games_played"] >= 0.75 * games_df["team_games"]
    )

    # ------------------------------------------------
    # 3. Aggregate per-game stats
    # ------------------------------------------------
    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    numeric_cols = [c for c in numeric_cols if c not in ["gameid", "date"]]

    stats_df = (
        df.groupby(["player", "team"])[numeric_cols]
        .mean()
        .reset_index()
    )

    stats_df = stats_df.merge(
        games_df[["player", "team", "games_played", "qualifies_games"]],
        on=["player", "team"],
        how="left"
    )

    # ------------------------------------------------
    # 4. Made-per-game qualification rules
    # ------------------------------------------------
    makes = (
        df.groupby(["player", "team"])
        .agg(
            FGM=("fgm", "sum"),
            FG2M=("fg2m", "sum"),
            FG3M=("fg3m", "sum"),
            FTM=("ftm", "sum")
        )
        .reset_index()
    )

    stats_df = stats_df.merge(makes, on=["player", "team"], how="left")

    if stat in ["fg2_pct", "fg3_pct"]:
        stats_df = stats_df[
            stats_df[f"fg{stat[2]}m"] >= 2 * stats_df["games_played"]
        ]

    elif stat == "ft_pct":
        stats_df = stats_df[
            stats_df["ftm"] >= 2 * stats_df["games_played"]
        ]

    elif stat == "fg_pct":
        stats_df = stats_df[
            stats_df["fgm"] >= 4 * stats_df["games_played"]
        ]

    # ------------------------------------------------
    # 5. Apply games-played rule
    # ------------------------------------------------
    stats_df = stats_df[stats_df["qualifies_games"]]

    # ------------------------------------------------
    # 6. Rank leaders
    # ------------------------------------------------
    leaders = (
        stats_df
        .sort_values(stat, ascending=False)
        .head(int(n))
        .reset_index(drop=True)
    )

    leaders["rank"] = np.arange(1, len(leaders) + 1)
    leaders[stat] = leaders[stat].map("{:.3f}".format)

    return leaders[["rank", "player", "team", stat]]

def get_conference_player_pg_stat_rank(conference, stat, player):
    if stat not in players_box_df.columns:
        raise Exception(f"{stat} is not valid")

    id_cols = ["player", "team", "conference"]

    df = players_box_df[players_box_df.conference.eq(conference)].copy()

    # -----------------------------
    # 1. Games played per player
    # -----------------------------
    games_df = (
        df.groupby(["player", "team"])
        .agg(games_played=("gameid", "nunique"))
        .reset_index()
    )

    # -----------------------------
    # 2. Team games played
    # -----------------------------
    team_games_df = (
        df.groupby("team")
        .agg(team_games=("gameid", "nunique"))
        .reset_index()
    )

    games_df = games_df.merge(team_games_df, on="team", how="left")

    # 75% team games rule
    games_df["qualifies_games"] = (
        games_df["games_played"] >= 0.75 * games_df["team_games"]
    )

    # -----------------------------
    # 3. Aggregate player stats
    # -----------------------------
    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    numeric_cols = [c for c in numeric_cols if c not in ["gameid", "date"]]

    stats_df = (
        df[id_cols + numeric_cols]
        .groupby(id_cols)[numeric_cols]
        .mean()
        .reset_index()
    )

    stats_df = stats_df.merge(
        games_df[["player", "team", "games_played", "qualifies_games"]],
        on=["player", "team"],
        how="left"
    )

    # -----------------------------
    # 4. Percentage qualification rules
    # -----------------------------


    if stat == "fg_pct":
        total_fgm = df.groupby(["player", "team"])["fgm"].sum().reset_index()
        stats_df = stats_df.drop("fgm", axis=1).merge(total_fgm, on=["player", "team"])
        stats_df = stats_df[
            stats_df["fgm"] >= 4 * stats_df["games_played"]
        ]

    elif stat == "fg2_pct":
        total_tpm = df.groupby(["player", "team"])["fg2m"].sum().reset_index()
        stats_df = stats_df.drop("fg2m", axis=1).merge(total_tpm, on=["player", "team"])
        stats_df = stats_df[
            stats_df["fg2m"] >= 2 * stats_df["games_played"]
        ]

    elif stat == "fg3_pct":
        total_tpm = df.groupby(["player", "team"])["fg3m"].sum().reset_index()
        stats_df = stats_df.drop("fg3m", axis=1).merge(total_tpm, on=["player", "team"])
        stats_df = stats_df[
            stats_df["fg3m"] >= 2 * stats_df["games_played"]
        ]

    elif stat == "ft_pct":
        total_ftm = df.groupby(["player", "team"])["ftm"].sum().reset_index()
        stats_df = stats_df.drop("ftm", axis=1).merge(total_ftm, on=["player", "team"])
        stats_df = stats_df[
            stats_df["ftm"] >= 2 * stats_df["games_played"]
        ]

    # -----------------------------
    # 5. Apply games-played filter
    # -----------------------------
    stats_df = stats_df[stats_df["qualifies_games"]]

    # -----------------------------
    # 6. Rank leaders
    # -----------------------------
    leaders = (
        stats_df
        .sort_values(stat, ascending=False)
        .reset_index(drop=True)
    )

    leaders["rank"] = np.arange(1, len(leaders) + 1)
    leaders[stat] = leaders[stat].map("{:.3f}".format)

    rank = leaders[leaders.player == player]["rank"].iloc[0]
    return jsonify({"Rank" : int(rank)})

def get_national_player_pg_stat_leaders(stat, n=25):
    if stat not in players_box_df.columns:
        raise Exception("stat is not valid")

    id_cols = ["player", "team", "conference"]

    df = players_box_df[players_box_df.conference.ne("Not D1")].copy()

    # -----------------------------
    # 1. Games played per player
    # -----------------------------
    games_df = (
        df.groupby(["player", "team"])
        .agg(games_played=("gameid", "nunique"))
        .reset_index()
    )

    # -----------------------------
    # 2. Team games played
    # -----------------------------
    team_games_df = (
        df.groupby("team")
        .agg(team_games=("gameid", "nunique"))
        .reset_index()
    )

    games_df = games_df.merge(team_games_df, on="team", how="left")

    # 75% team games rule
    games_df["qualifies_games"] = (
        games_df["games_played"] >= 0.75 * games_df["team_games"]
    )

    # -----------------------------
    # 3. Aggregate player stats
    # -----------------------------
    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    numeric_cols = [c for c in numeric_cols if c not in ["gameid", "date"]]

    stats_df = (
        df[id_cols + numeric_cols]
        .groupby(id_cols)[numeric_cols]
        .mean()
        .reset_index()
    )

    stats_df = stats_df.merge(
        games_df[["player", "team", "games_played", "qualifies_games"]],
        on=["player", "team"],
        how="left"
    )

    # -----------------------------
    # 4. Percentage qualification rules
    # -----------------------------


    if stat == "fg_pct":
        total_fgm = df.groupby(["player", "team"])["fgm"].sum().reset_index()
        stats_df = stats_df.drop("fgm", axis=1).merge(total_fgm, on=["player", "team"])
        print(stats_df.columns)
        stats_df = stats_df[
            stats_df["fgm"] >= 4 * stats_df["games_played"]
        ]

    elif stat == "fg2_pct":
        total_tpm = df.groupby(["player", "team"])["fg2m"].sum().reset_index()
        stats_df = stats_df.drop("fg2m", axis=1).merge(total_tpm, on=["player", "team"])
        stats_df = stats_df[
            stats_df["fg2m"] >= 2 * stats_df["games_played"]
        ]

    elif stat == "fg3_pct":
        total_tpm = df.groupby(["player", "team"])["fg3m"].sum().reset_index()
        stats_df = stats_df.drop("fg3m", axis=1).merge(total_tpm, on=["player", "team"])
        stats_df = stats_df[
            stats_df["fg3m"] >= 2 * stats_df["games_played"]
        ]

    elif stat == "ft_pct":
        total_ftm = df.groupby(["player", "team"])["ftm"].sum().reset_index()
        stats_df = stats_df.drop("ftm", axis=1).merge(total_ftm, on=["player", "team"])
        stats_df = stats_df[
            stats_df["ftm"] >= 2 * stats_df["games_played"]
        ]

    # -----------------------------
    # 5. Apply games-played filter
    # -----------------------------
    stats_df = stats_df[stats_df["qualifies_games"]]

    # -----------------------------
    # 6. Rank leaders
    # -----------------------------
    leaders = (
        stats_df
        .sort_values(stat, ascending=False)
        .head(n)
        .reset_index(drop=True)
    )

    leaders["rank"] = np.arange(1, len(leaders) + 1)
    leaders[stat] = leaders[stat].map("{:.3f}".format)

    return leaders[["rank", "player", "team", "conference", stat]]

def get_national_player_pg_stat_rank(stat, player):
    if stat not in players_box_df.columns:
        raise Exception("stat is not valid")

    id_cols = ["player", "team", "conference"]

    df = players_box_df[players_box_df.conference.ne("Not D1")].copy()

    # -----------------------------
    # 1. Games played per player
    # -----------------------------
    games_df = (
        df.groupby(["player", "team"])
        .agg(games_played=("gameid", "nunique"))
        .reset_index()
    )

    # -----------------------------
    # 2. Team games played
    # -----------------------------
    team_games_df = (
        df.groupby("team")
        .agg(team_games=("gameid", "nunique"))
        .reset_index()
    )

    games_df = games_df.merge(team_games_df, on="team", how="left")

    # 75% team games rule
    games_df["qualifies_games"] = (
        games_df["games_played"] >= 0.75 * games_df["team_games"]
    )

    # -----------------------------
    # 3. Aggregate player stats
    # -----------------------------
    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    numeric_cols = [c for c in numeric_cols if c not in ["gameid", "date"]]

    stats_df = (
        df[id_cols + numeric_cols]
        .groupby(id_cols)[numeric_cols]
        .mean()
        .reset_index()
    )

    stats_df = stats_df.merge(
        games_df[["player", "team", "games_played", "qualifies_games"]],
        on=["player", "team"],
        how="left"
    )

    # -----------------------------
    # 4. Percentage qualification rules
    # -----------------------------


    if stat == "fg_pct":
        total_fgm = df.groupby(["player", "team"])["fgm"].sum().reset_index()
        stats_df = stats_df.drop("fgm", axis=1).merge(total_fgm, on=["player", "team"])
        print(stats_df.columns)
        stats_df = stats_df[
            stats_df["fgm"] >= 4 * stats_df["games_played"]
        ]

    elif stat == "fg2_pct":
        total_tpm = df.groupby(["player", "team"])["fg2m"].sum().reset_index()
        stats_df = stats_df.drop("fg2m", axis=1).merge(total_tpm, on=["player", "team"])
        stats_df = stats_df[
            stats_df["fg2m"] >= 2 * stats_df["games_played"]
        ]

    elif stat == "fg3_pct":
        total_tpm = df.groupby(["player", "team"])["fg3m"].sum().reset_index()
        stats_df = stats_df.drop("fg3m", axis=1).merge(total_tpm, on=["player", "team"])
        stats_df = stats_df[
            stats_df["fg3m"] >= 2 * stats_df["games_played"]
        ]

    elif stat == "ft_pct":
        total_ftm = df.groupby(["player", "team"])["ftm"].sum().reset_index()
        stats_df = stats_df.drop("ftm", axis=1).merge(total_ftm, on=["player", "team"])
        stats_df = stats_df[
            stats_df["ftm"] >= 2 * stats_df["games_played"]
        ]

    # -----------------------------
    # 5. Apply games-played filter
    # -----------------------------
    stats_df = stats_df[stats_df["qualifies_games"]]

    # -----------------------------
    # 6. Rank leaders
    # -----------------------------
    leaders = (
        stats_df
        .sort_values(stat, ascending=False)
        .reset_index(drop=True)
    )

    leaders["rank"] = np.arange(1, len(leaders) + 1)
    leaders[stat] = leaders[stat].map("{:.3f}".format)

    rank = leaders[leaders.player == player]["rank"].iloc[0]
    return jsonify({"Rank" : int(rank)})

def get_team_pg_stats(team):
    
    if team not in matchups_df["team"].unique():
        raise Exception("team not found in data")
    
    team_games = (
        matchups_df[matchups_df.team == team]
        [["team", "team_conference", "team_win", "team_loss", "team_pts", "team_trb", "team_orb", "team_drb", 
          "team_ast", "team_stl", "team_blk", "team_tov", "team_fga", "team_fgm", "team_fg2m", "team_fg2a", "team_fg3m", "team_fg3a", 
          "team_ftm", "team_fta", "team_pf", "team_poss", "team_ortg", "team_drtg", "team_netrtg", "opponent_pts", 
          "opponent_trb", "opponent_orb", "opponent_drb", "opponent_ast", "opponent_stl", 
          "opponent_blk", "opponent_tov", "opponent_fga", "opponent_fgm", "opponent_fg2m", 
          "opponent_fg2a", "opponent_fg3m", "opponent_fg3a", "opponent_ftm", "opponent_fta", "opponent_pf"
        ]]
    )
                                            
    numeric_cols = [i for i in team_games.columns if i not in ["team", "team_conference"]]

    df = (
        team_games
        .groupby(["team", "team_conference"])[numeric_cols]
        .agg({**{k : "mean" for k in numeric_cols}, "team_win" : "sum", "team_loss" : "sum"})
        .reset_index()
    )

    df = df.rename(columns={
        "opponent_pts" : "pts_allowed",
        "opponent_trb" : "trb_allowed",
        "opponent_orb" : "orb_allowed",
        "opponent_drb" : "drb_allowed",
        "opponent_ast" : "ast_allowed",
        "opponent_stl" : "stl_allowed",
        "opponent_blk" : "shots_blocked",
        "opponent_tov" : "forced_tov",
        "opponent_fga" : "fga_allowed",
        "opponent_fgm" : "fgm_allowed", 
        "opponent_fg2m" : "fg2m_allowed",
        "opponent_fg2a" : "fg2a_allowed",
        "opponent_fg3m" : "fg3m_allowed",
        "opponent_fg3a" : "fg3a_allowed",
        "opponent_ftm" : "ftm_allowed",
        "opponent_fta" : "fta_allowed",
        "opponent_pf" : "pf_drawn",
        "team_win" : "W", 
        "team_loss" : "L"
    })

    df["team_fg2_pct"] = df["team_fg2m"] / df["team_fg2a"]
    df["team_fg3_pct"] = df["team_fg3m"] / df["team_fg3a"]
    df["team_ft_pct"] = df["team_ftm"] / df["team_fta"]
    df["team_fg_pct"] = df["team_fgm"] / df["team_fga"]
    df["fg2_pct_allowed"] = df["fg2m_allowed"] / df["fg2a_allowed"]
    df["fg3_pct_allowed"] = df["fg3m_allowed"] / df["fg3a_allowed"]
    df["ft_pct_allowed"] = df["ftm_allowed"] / df["fta_allowed"]
    df["fg_pct_allowed"] = df["fgm_allowed"] / df["fga_allowed"]
    df["gp"] = [len(team_games)]

    numeric_cols = [i for i in df.columns if i not in ["team", "team_conference", "W", "L", "gp"]]

    df[numeric_cols] = df[numeric_cols].map("{:.3f}".format)

    return df

def get_teams_pg_stats(teams):
    df = pd.DataFrame()
    for team in teams:
        df = pd.concat([df, get_team_pg_stats(team)], axis=0)
    return df 

def compare_teams(df, teams, stats=None, figsize=(14,6), title=None, colors=None):
    for t in teams:
        if t not in df["team"].values:
            raise ValueError(f"{t} not found in DataFrame")

    sub = df[df["team"].isin(teams)].set_index("team")

    if stats is None:
        stats = sub.select_dtypes(include=[np.number]).columns.tolist()

    pct_stats = [col for col in stats if col.endswith("%") or col.endswith("_pct") or col.endswith("%_allowed") or col.endswith("_pct_allowed")]
    raw_stats = [col for col in stats if col not in pct_stats]

    def _plot_category(stat_list, title_suffix, colors):
        if len(stat_list) == 0:
            return

        values = sub[stat_list].astype(float)
        n_teams = len(teams)

        x = np.arange(len(stat_list))
        width = 0.8 / n_teams

        fig, ax = plt.subplots(figsize=figsize)

        bar_containers = []

        if not colors:
            colors = ["lightblue", "pink", "violet", "pastelgreen"]
        for i, team in enumerate(teams):
            bars = ax.bar(
                x + (i - n_teams/2) * width + width/2,
                values.loc[team],
                width,
                label=team,
                ec="black",
                color=colors[i]
            )
            bar_containers.append((team, bars))

        max_val = values.max().max()
        ax.set_ylim(0, 1.5 * max_val)

        for team, bars in bar_containers:
            for rect in bars:
                height = rect.get_height()
                ax.annotate(
                    f"{height:.2f}",  
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 4),  
                    textcoords="offset points",
                    ha="center",
                    va="bottom",
                    fontsize=9
                )

        ax.set_xticks(x)
        ax.set_xticklabels(stat_list, rotation=45, ha="right")
        ax.set_ylabel("Stat Value")

        final_title = (title or "Team Comparison") + f" — {title_suffix}"
        ax.set_title(final_title)
        ax.legend()

        plt.tight_layout()
        plt.show()

    _plot_category(raw_stats, "Raw Stats", colors)

    _plot_category(pct_stats, "Percentage Stats", colors)

def get_conference_team_pg_stat_leaders(conference, stat):
    if conference not in conferences:
        raise Exception("conference or stat is not valid")

    teams = matchups_df[matchups_df.team_conference == conference]["team"].unique()

    df = get_teams_pg_stats(teams)

    df[stat] = df[stat].astype("float")

    df = (
        df
        .sort_values(stat, ascending=("_allowed" in stat or stat == "shots_blocked"))
        .loc[:, ["team", stat]]
    )
    df["rank"] = np.arange(1, len(df) + 1)

    return df[["rank", "team", stat]]

def get_conference_team_pg_stat_rank(conference, team, stat):
    id_cols = ["team", "team_conference"]

    numeric_cols = matchups_df.select_dtypes(include="number").columns.tolist()

    numeric_cols = [c for c in numeric_cols if c not in ["gameid", "date"]]

    df = (
        matchups_df[matchups_df.team_conference == conference][id_cols + numeric_cols]
        .groupby(["team", "team_conference"])[numeric_cols]
        .mean()
        .rename(columns={
            "opponent_pts" : "pts_allowed",
            "opponent_trb" : "trb_allowed",
            "opponent_orb" : "orb_allowed",
            "opponent_drb" : "drb_allowed",
            "opponent_ast" : "ast_allowed",
            "opponent_stl" : "stl_allowed",
            "opponent_blk" : "shots_blocked",
            "opponent_tov" : "forced_tov",
            "opponent_fga" : "fga_allowed",
            "opponent_fgm" : "fgm_allowed", 
            "opponent_fg2m" : "fg2m_allowed",
            "opponent_fg2a" : "fg2a_allowed",
            "opponent_fg3m" : "fg3m_allowed",
            "opponent_fg3a" : "fg3a_allowed",
            "opponent_ftm" : "ftm_allowed",
            "opponent_fta" : "fta_allowed",
            "opponent_pf" : "pf_drawn",
            "team_win" : "W", 
            "team_loss" : "L"
        })
        .reset_index()
    )

    df["team_fg2_pct"] = df["team_fg2m"] / df["team_fg2a"]
    df["team_fg3_pct"] = df["team_fg3m"] / df["team_fg3a"]
    df["team_ft_pct"] = df["team_ftm"] / df["team_fta"]
    df["team_fg_pct"] = df["team_fgm"] / df["team_fga"]
    df["fg2_pct_allowed"] = df["fg2m_allowed"] / df["fg2a_allowed"]
    df["fg3_pct_allowed"] = df["fg3m_allowed"] / df["fg3a_allowed"]
    df["ft_pct_allowed"] = df["ftm_allowed"] / df["fta_allowed"]
    df["fg_pct_allowed"] = df["fgm_allowed"] / df["fga_allowed"]
    df = df.sort_values(stat, ascending=("_allowed" in stat or 'blocked' in stat))
    df["rank"] = np.arange(1, len(df)+1)

    found_rank = df[df.team == team]["rank"].iloc[0]

    return {"Rank" : int(found_rank)}

def get_national_team_pg_stat_leaders(stat, n=25, find_rank=None):
    id_cols = ["team", "team_conference"]

    numeric_cols = matchups_df.select_dtypes(include="number").columns.tolist()

    numeric_cols = [c for c in numeric_cols if c not in ["gameid", "date"]]

    df = (
        matchups_df[matchups_df.team_conference != "Not D1"][id_cols + numeric_cols]
        .groupby(["team", "team_conference"])[numeric_cols]
        .mean()
        .rename(columns={
            "opponent_pts" : "pts_allowed",
            "opponent_trb" : "trb_allowed",
            "opponent_orb" : "orb_allowed",
            "opponent_drb" : "drb_allowed",
            "opponent_ast" : "ast_allowed",
            "opponent_stl" : "stl_allowed",
            "opponent_blk" : "shots_blocked",
            "opponent_tov" : "forced_tov",
            "opponent_fga" : "fga_allowed",
            "opponent_fgm" : "fgm_allowed", 
            "opponent_fg2m" : "fg2m_allowed",
            "opponent_fg2a" : "fg2a_allowed",
            "opponent_fg3m" : "fg3m_allowed",
            "opponent_fg3a" : "fg3a_allowed",
            "opponent_ftm" : "ftm_allowed",
            "opponent_fta" : "fta_allowed",
            "opponent_pf" : "pf_drawn",
            "team_win" : "W", 
            "team_loss" : "L"
        })
        .reset_index()
    )

    df["team_fg2_pct"] = df["team_fg2m"] / df["team_fg2a"]
    df["team_fg3_pct"] = df["team_fg3m"] / df["team_fg3a"]
    df["team_ft_pct"] = df["team_ftm"] / df["team_fta"]
    df["team_fg_pct"] = df["team_fgm"] / df["team_fga"]
    df["fg2_pct_allowed"] = df["fg2m_allowed"] / df["fg2a_allowed"]
    df["fg3_pct_allowed"] = df["fg3m_allowed"] / df["fg3a_allowed"]
    df["ft_pct_allowed"] = df["ftm_allowed"] / df["fta_allowed"]
    df["fg_pct_allowed"] = df["fgm_allowed"] / df["fga_allowed"]
    df = df.sort_values(stat, ascending=("_allowed" in stat or "_blocked" in stat))
    df["rank"] = np.arange(1, len(df)+1)
    sliced_df = df.loc[:, ["rank", "team", "team_conference", stat]].head(n)
    sliced_df[stat] = df[stat].map("{:.3f}".format)
    
    if find_rank and not isinstance(find_rank, list):
        found_rank = df[df.team == find_rank]["rank"].iloc[0]
        print(f"{stat} rank for {find_rank}: {found_rank}")
    elif find_rank and isinstance(find_rank, list):
        for team in find_rank:
            found_rank = df[df.team == team]["rank"].iloc[0]
            print(f"{stat} rank for {team}: {found_rank}")

    return sliced_df.reset_index(drop=True)

def get_national_team_pg_stat_rank(stat, team):
    id_cols = ["team", "team_conference"]

    numeric_cols = matchups_df.select_dtypes(include="number").columns.tolist()

    numeric_cols = [c for c in numeric_cols if c not in ["gameid", "date"]]

    df = (
        matchups_df[matchups_df.team_conference != "Not D1"][id_cols + numeric_cols]
        .groupby(["team", "team_conference"])[numeric_cols]
        .mean()
        .rename(columns={
            "opponent_pts" : "pts_allowed",
            "opponent_trb" : "trb_allowed",
            "opponent_orb" : "orb_allowed",
            "opponent_drb" : "drb_allowed",
            "opponent_ast" : "ast_allowed",
            "opponent_stl" : "stl_allowed",
            "opponent_blk" : "shots_blocked",
            "opponent_tov" : "forced_tov",
            "opponent_fga" : "fga_allowed",
            "opponent_fgm" : "fgm_allowed", 
            "opponent_fg2m" : "fg2m_allowed",
            "opponent_fg2a" : "fg2a_allowed",
            "opponent_fg3m" : "fg3m_allowed",
            "opponent_fg3a" : "fg3a_allowed",
            "opponent_ftm" : "ftm_allowed",
            "opponent_fta" : "fta_allowed",
            "opponent_pf" : "pf_drawn",
            "team_win" : "W", 
            "team_loss" : "L"
        })
        .reset_index()
    )

    df["team_fg2_pct"] = df["team_fg2m"] / df["team_fg2a"]
    df["team_fg3_pct"] = df["team_fg3m"] / df["team_fg3a"]
    df["team_ft_pct"] = df["team_ftm"] / df["team_fta"]
    df["team_fg_pct"] = df["team_fgm"] / df["team_fga"]
    df["fg2_pct_allowed"] = df["fg2m_allowed"] / df["fg2a_allowed"]
    df["fg3_pct_allowed"] = df["fg3m_allowed"] / df["fg3a_allowed"]
    df["ft_pct_allowed"] = df["ftm_allowed"] / df["fta_allowed"]
    df["fg_pct_allowed"] = df["fgm_allowed"] / df["fga_allowed"]
    df = df.sort_values(stat, ascending=("_allowed" in stat or "_blocked" in stat))
    df["rank"] = np.arange(1, len(df)+1)

    found_rank = df[df.team == team]["rank"].iloc[0]

    return jsonify({"Rank" : int(found_rank)})

def compute_all_team_ratings():
    all_teams = matchups_df["team"].unique()

    NatAvgOE = matchups_df["team_ortg"].mean()
    NatAvgDE = matchups_df["team_drtg"].mean()

    team_ratings = {
        t: {"AdjO": NatAvgOE, "AdjD": NatAvgDE}
        for t in all_teams
    }

    def compute_adjusted_efficiencies():
        new = {}
        for t in all_teams:
            games = matchups_df[matchups_df.team == t]

            adjO = []
            adjD = []

            for _, g in games.iterrows():
                opp = g.opponent
                if opp not in team_ratings:
                    continue

                adjO.append(g.team_ortg * (NatAvgOE / team_ratings[opp]["AdjD"]))
                adjD.append(g.team_drtg * (NatAvgOE / team_ratings[opp]["AdjO"]))

            new[t] = {
                "AdjO": np.mean(adjO) if adjO else NatAvgOE,
                "AdjD": np.mean(adjD) if adjD else NatAvgDE
            }
        return new

    # iterate 25–30 times
    for _ in range(30):
        team_ratings = compute_adjusted_efficiencies()

    # Add Pyth rating
    x = 10.25
    for t in all_teams:
        O = team_ratings[t]["AdjO"]
        D = team_ratings[t]["AdjD"]
        team_ratings[t]["Pyth"] = (O**x) / (O**x + D**x)

    return team_ratings

def get_team_sos(team, team_ratings):
    x = 10.25

    opps = matchups_df.loc[matchups_df.team == team, "opponent"]
    adjO = [team_ratings[o]["AdjO"] for o in opps if o in team_ratings]
    adjD = [team_ratings[o]["AdjD"] for o in opps if o in team_ratings]

    sos_AdjO = np.mean(adjO)
    sos_AdjD = np.mean(adjD)
    sos_Pyth = (sos_AdjO**x) / (sos_AdjO**x + sos_AdjD**x)

    return float(sos_Pyth)

def get_national_sos_rankings(team_ratings, n=25, top_n_net=50, sort_by="SOS", find_rank=None):

    df = matchups_df.loc[matchups_df.team_conference != "Not D1", ["team", "team_conference"]].drop_duplicates()

    df["sos"] = df["team"].apply(lambda t: get_team_sos(t, team_ratings))

    rating_df = pd.DataFrame([
        {"Team": t,
         "AdjEM": team_ratings[t]["AdjO"] - team_ratings[t]["AdjD"]}
        for t in team_ratings
    ])

    rating_df = rating_df.sort_values("AdjEM", ascending=False)
    topN_teams = set(rating_df["Team"].iloc[:top_n_net])

    def count_topN_opponents(team):
        opponents = matchups_df.loc[matchups_df.team == team, "opponent"]
        return sum(o in topN_teams for o in opponents)

    df[f"top{top_n_net}_opp_count"] = df["team"].apply(count_topN_opponents)


    df = df.sort_values("sos" if sort_by=="SOS" else f"Top{top_n_net}_Opp_Count", ascending=False)
    df["rank"] = np.arange(1, len(df) + 1)

    if find_rank and isinstance(find_rank, list):
        found_ranks = {}
        for team in find_rank:
            found_rank = df[df.team == team]["rank"].iloc[0]
            found_ranks[team] = found_rank
            
        return df[["rank", "team", "team_conference", "sos", f"top{top_n_net}_opp_count"]].iloc[:n].reset_index(drop=True), found_ranks
    
    return df[["rank", "team", "team_conference", "sos", f"top{top_n_net}_opp_count"]].iloc[:n].reset_index(drop=True)

def player_pg_performance_filter(filter_dict):
    id_cols = ["player", "team", "conference"]
    
    # numeric columns excluding GameID and Date
    numeric_cols = players_box_df.select_dtypes(include="number").columns.tolist()
    numeric_cols = [c for c in numeric_cols if c not in ["gameid", "date"]]

    # --- Compute GS (games started) per player BEFORE groupby ---
    gs = (
        players_box_df
        .groupby("player")["role"]
        .apply(lambda x: (x == "Starter").sum())
        .rename("gs")
    )

    # --- Build per-game averages ---
    df = (
        players_box_df[id_cols + numeric_cols]
        .groupby(["player", "team", "conference"])[numeric_cols]
        .mean()
        .reset_index()
    )

    # Add GS as a column
    df = df.merge(gs, on="player", how="left")

    # --- Build boolean mask ---
    mask = pd.Series(True, index=df.index)

    for col, condition in filter_dict.items():
        op, value = condition
        if op == ">":
            mask &= df[col] > value
        elif op == ">=":
            mask &= df[col] >= value
        elif op == "<":
            mask &= df[col] < value
        elif op == "<=":
            mask &= df[col] <= value
        elif op == "==":
            mask &= df[col] == value
        elif op == "!=":
            mask &= df[col] != value
        else:
            raise ValueError(f"Unsupported operator: {op}")

    df = df[mask].reset_index(drop=True)

    # Format numeric columns
    df[numeric_cols] = df[numeric_cols].map("{:.3f}".format)

    return df[["player", "team", "conference"] + list(filter_dict.keys())].sort_values("team")

def get_team_player_pg_stat_leaders(team, stat):
    if team not in matchups_df["team"].unique() or stat not in players_box_df.columns:
        raise Exception("Team or stat is not valid")

    players = players_box_df[players_box_df.team == team]["player"].unique()

    df = get_players_pg_stats(players)

    df[stat] = df[stat].astype("float")

    df = (
        df
        .sort_values(stat, ascending=False)
        .loc[:, ["player", stat]]
    )
    df[stat] = df[stat].map("{:.3f}".format)
    df["rank"] = np.arange(1, len(df) + 1)

    return df[["rank", "player", stat]]

def get_team_player_pg_stat_rank(team, stat, player):
    if stat not in players_box_df.columns:
        raise Exception("stat is not valid")

    id_cols = ["player", "team", "conference"]

    df = players_box_df[players_box_df.team.eq(team)].copy()

    # -----------------------------
    # 1. Games played per player
    # -----------------------------
    games_df = (
        df.groupby(["player", "team"])
        .agg(games_played=("gameid", "nunique"))
        .reset_index()
    )

    # -----------------------------
    # 2. Team games played
    # -----------------------------
    team_games_df = (
        df.groupby("team")
        .agg(team_games=("gameid", "nunique"))
        .reset_index()
    )

    games_df = games_df.merge(team_games_df, on="team", how="left")

    # 75% team games rule
    games_df["qualifies_games"] = (
        games_df["games_played"] >= 0.75 * games_df["team_games"]
    )

    # -----------------------------
    # 3. Aggregate player stats
    # -----------------------------
    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    numeric_cols = [c for c in numeric_cols if c not in ["gameid", "date"]]

    stats_df = (
        df[id_cols + numeric_cols]
        .groupby(id_cols)[numeric_cols]
        .mean()
        .reset_index()
    )

    stats_df = stats_df.merge(
        games_df[["player", "team", "games_played", "qualifies_games"]],
        on=["player", "team"],
        how="left"
    )

    # -----------------------------
    # 4. Percentage qualification rules
    # -----------------------------


    if stat == "fg_pct":
        total_fgm = df.groupby(["player", "team"])["fgm"].sum().reset_index()
        stats_df = stats_df.drop("fgm", axis=1).merge(total_fgm, on=["player", "team"])
        stats_df = stats_df[
            stats_df["fgm"] >= 4 * stats_df["games_played"]
        ]

    elif stat == "fg2_pct":
        total_tpm = df.groupby(["player", "team"])["fg2m"].sum().reset_index()
        stats_df = stats_df.drop("fg2m", axis=1).merge(total_tpm, on=["player", "team"])
        stats_df = stats_df[
            stats_df["fg2m"] >= 2 * stats_df["games_played"]
        ]

    elif stat == "fg3_pct":
        total_tpm = df.groupby(["player", "team"])["fg3m"].sum().reset_index()
        stats_df = stats_df.drop("fg3m", axis=1).merge(total_tpm, on=["player", "team"])
        stats_df = stats_df[
            stats_df["fg3m"] >= 2 * stats_df["games_played"]
        ]

    elif stat == "ft_pct":
        total_ftm = df.groupby(["player", "team"])["ftm"].sum().reset_index()
        stats_df = stats_df.drop("ftm", axis=1).merge(total_ftm, on=["player", "team"])
        stats_df = stats_df[
            stats_df["ftm"] >= 2 * stats_df["games_played"]
        ]

    # -----------------------------
    # 5. Apply games-played filter
    # -----------------------------
    stats_df = stats_df[stats_df["qualifies_games"]]

    # -----------------------------
    # 6. Rank leaders
    # -----------------------------
    leaders = (
        stats_df
        .sort_values(stat, ascending=False)
        .reset_index(drop=True)
    )

    leaders["rank"] = np.arange(1, len(leaders) + 1)
    leaders[stat] = leaders[stat].map("{:.3f}".format)

    rank = leaders[leaders.player == player]["rank"].iloc[0]
    return {"Rank" : int(rank)}


In [14]:
from io import StringIO
def get_NET_ratings():
    url = "https://www.ncaa.com/rankings/basketball-men/d1/ncaa-mens-basketball-net-rankings"
    r = requests.get(url)
    soup = BeautifulSoup(r.content, 'html.parser')
    table = soup.find('table')
    df = pd.read_html(StringIO(str(table)))[0]
    df["Quad 1 W"] = df["Quad 1"].str.extract(r'(\d+)-\d+').astype(int)
    df["Quad 1 L"] = df["Quad 1"].str.extract(r'\d+-(\d+)').astype(int)
    df["Quad 2 W"] = df["Quad 2"].str.extract(r'(\d+)-\d+').astype(int) 
    df["Quad 2 L"] = df["Quad 2"].str.extract(r'\d+-(\d+)').astype(int)
    df["Quad 3 W"] = df["Quad 3"].str.extract(r'(\d+)-\d+').astype(int) 
    df["Quad 3 L"] = df["Quad 3"].str.extract(r'\d+-(\d+)').astype(int)
    df["Quad 4 W"] = df["Quad 4"].str.extract(r'(\d+)-\d+').astype(int) 
    df["Quad 4 L"] = df["Quad 4"].str.extract(r'\d+-(\d+)').astype(int)
    df["W"] = df["Record"].str.extract(r'(\d+)-\d+').astype(int)
    df["L"] = df["Record"].str.extract(r'\d+-(\d+)').astype(int)
    df["Road W"] = df["Road"].str.extract(r'(\d+)-\d+').astype(int)
    df["Road L"] = df["Road"].str.extract(r'\d+-(\d+)').astype(int)
    df["Home W"] = df["Home"].str.extract(r'(\d+)-\d+').astype(int)
    df["Home L"] = df["Home"].str.extract(r'\d+-(\d+)').astype(int)
    df["Neutral W"] = df["Neutral"].str.extract(r'(\d+)-\d+').astype(int)
    df["Neutral L"] = df["Neutral"].str.extract(r'\d+-(\d+)').astype(int)
    df["Non-Div I W"] = df["Non-Div I"].str.extract(r'(\d+)-\d+').astype(int)
    df["Non-Div I L"] = df["Non-Div I"].str.extract(r'\d+-(\d+)').astype(int)
    df.drop(columns=["Quad 1", "Quad 2", "Quad 3", "Quad 4", "Home", "Road", "Neutral", "Record", "Non-Div I"], inplace=True)
    df.to_csv("../backend/data/NET_ratings.csv", index=False)

get_NET_ratings()